In [ ]:
from rank_bm25 import BM25Okapi
import pandas as pd
import re
from rapidfuzz import process, fuzz
import json

# --- Load and build full address string ---
detail   = pd.read_csv("GNAF/VIC_ADDRESS_DETAIL_psv.psv", sep="|", dtype='str', usecols=['ADDRESS_DETAIL_PID','BUILDING_NAME','LOT_NUMBER',
       'LOT_NUMBER_SUFFIX', 'FLAT_TYPE_CODE', 'FLAT_NUMBER_PREFIX',
       'FLAT_NUMBER', 'FLAT_NUMBER_SUFFIX', 'LEVEL_TYPE_CODE',
       'LEVEL_NUMBER_PREFIX', 'LEVEL_NUMBER', 'LEVEL_NUMBER_SUFFIX',
       'NUMBER_FIRST_PREFIX', 'NUMBER_FIRST', 'NUMBER_FIRST_SUFFIX',
       'NUMBER_LAST_PREFIX', 'NUMBER_LAST', 'NUMBER_LAST_SUFFIX','POSTCODE','STREET_LOCALITY_PID'])
street   = pd.read_csv("GNAF/VIC_STREET_LOCALITY_psv.psv", sep="|", dtype='str', usecols=['STREET_LOCALITY_PID','STREET_NAME','STREET_TYPE_CODE','LOCALITY_PID','GNAF_STREET_PID'])
locality = pd.read_csv("GNAF/VIC_LOCALITY_psv.psv", sep="|", dtype='str', usecols=['LOCALITY_PID','LOCALITY_NAME','GNAF_LOCALITY_PID','STATE_PID'])
state    = pd.read_csv("GNAF/VIC_STATE_psv.psv", sep="|", dtype='str', usecols=['STATE_PID','STATE_NAME','STATE_ABBREVIATION'])
geocodedf  = pd.read_csv("GNAF/VIC_ADDRESS_DEFAULT_GEOCODE_psv.psv", sep="|", dtype='str', usecols=['ADDRESS_DETAIL_PID','LONGITUDE','LATITUDE'])


In [ ]:
# Transform input string to uppercase and split into tokens


def sanitize_input_string(input_string: str) -> str:

  STREET_ABBR = {
    "ST" :"STREET",
    "AV" :"AVENUE",
    "AVE" :"AVENUE",
    "RD" :"ROAD",
    "DR" :"DRIVE",
    "CRT" :"COURT",
    "PL" :"PLACE",
    "LN" :"LANE",
    "CIR" :"CIRCLE",
    "TCE" :"TERRACE",
    "BLVD" :"BOULEVARD",
    "PKWY" :"PARKWAY",
    "HWY" :"HIGHWAY",
    "SQ" :"SQUARE",
    "GR" :"GROVE",
    "PDE" :"PARADE",
    "CRES" :"CRESCENT"
  }

  addrinput_string = re.sub(r'[,.]', '', input_string)  # Remove punctuation except "/"
  upper_list = addrinput_string.upper().strip().strip().split()
  expanded_list = [STREET_ABBR.get(word, word) for word in upper_list]
  expanded_list

  santized_input_string = " ".join(expanded_list)
  print(f"Sanitized input string: {santized_input_string}")
  return santized_input_string

def search_addresses(query_string, bm25, n=10):
    tokens = query_string.upper().split()
    scores = bm25.get_scores(tokens)
    top_n_idx = scores.argsort()[::-1][:n]
    
    return top_n_idx
  
def create_output(match, score, input_string, dfaddress):
    match_idx = dfaddress[dfaddress['full_address'] == match].index[0]
    matched_address = dfaddress.loc[match_idx]
    matched_address = json.loads(matched_address.to_json())
    matched_address['score'] = score
    matched_address['InputAddress'] = input_string
    return matched_address
  
  
def geocode(input_string, dfaddress, bm25, n=10):
    santized_input_string = sanitize_input_string(input_string)
    search_results = search_addresses(santized_input_string, bm25, n=n)   
    match, score, _ = process.extractOne(santized_input_string, dfaddress.loc[search_results,'full_address'], scorer=fuzz.token_sort_ratio)
    matched_address = create_output(match, score, input_string, dfaddress)
    return matched_address

In [ ]:
dfaddress = detail.merge(
    street.merge(
        locality.merge(
            state, on="STATE_PID", how='left'), on="LOCALITY_PID", how='left'), on="STREET_LOCALITY_PID", how='left')

# --- Build full address string ---
dfaddress["full_address"] = (
    
    ## NEED TO COME BACK AND ADD BUILDING NAME, LOT NUMBER, FLAT NUMBER, LEVEL NUMBER, NUMBER LAST
    dfaddress["BUILDING_NAME"].fillna("").astype(str) + " " +
    dfaddress["LOT_NUMBER"].fillna("").astype(str) + " " +
    dfaddress["LOT_NUMBER_SUFFIX"].fillna("").astype(str) + " " +
    dfaddress["FLAT_TYPE_CODE"].fillna("").astype(str) + " " +
    dfaddress["FLAT_NUMBER_PREFIX"].fillna("").astype(str) + " " +
    dfaddress["FLAT_NUMBER"].fillna("").astype(str) + " " +
    dfaddress["FLAT_NUMBER_SUFFIX"].fillna("").astype(str) + " " +
    dfaddress["LEVEL_TYPE_CODE"].fillna("").astype(str) + " " +
    dfaddress["LEVEL_NUMBER_PREFIX"].fillna("").astype(str) + " " +
    dfaddress["LEVEL_NUMBER"].fillna("").astype(str) + " " +
    dfaddress["LEVEL_NUMBER_SUFFIX"].fillna("").astype(str) + " " +
    dfaddress["NUMBER_FIRST_PREFIX"].fillna("").astype(str) + " " +
    dfaddress["NUMBER_FIRST"].fillna("").astype(str) + " " +
    dfaddress["NUMBER_FIRST_SUFFIX"].fillna("").astype(str) + " " +
    dfaddress["NUMBER_LAST_PREFIX"].fillna("").astype(str) + " " +
    dfaddress["NUMBER_LAST"].fillna("").astype(str) + " " +
    dfaddress["NUMBER_LAST_SUFFIX"].fillna("").astype(str) + " " +
    dfaddress["STREET_NAME"].fillna("").astype(str) + " " +
    dfaddress["STREET_TYPE_CODE"].fillna("").astype(str) + " " +
    dfaddress["LOCALITY_NAME"].fillna("").astype(str) + " " +
    dfaddress["STATE_ABBREVIATION"].fillna("").astype(str) + " " +
    dfaddress["POSTCODE"].fillna("").astype(str)
)
dfaddress["full_address"].str.replace(r'\s+', ' ', regex=True).str.strip()
org_shape = dfaddress.shape
dfaddress = dfaddress[['ADDRESS_DETAIL_PID', 'full_address']].merge(
    geocodedf, on='ADDRESS_DETAIL_PID', how='left')

if dfaddress.shape[0] != org_shape[0]:
    print("Warning: Row count mismatch after merging geocoded data!")

(4344978, 30)
(4344978, 4)


In [ ]:
print("Building BM25 index...")
dfaddress["full_address_clean"] = dfaddress["full_address"].apply(sanitized_input_string)
bm25 = BM25Okapi(dfaddress["full_address_clean"].str.split().tolist())

Building BM25 index...


In [17]:
input_string = "Unit 5 16 Fladstone parade Glenroy 3040"

In [ ]:
geocode(input_string, dfaddress, bm25, n=10)

Sanitized input string: UNIT 5 16 FLADSTONE PARADE GLENROY 3040


{'ADDRESS_DETAIL_PID': 'GAVIC421714667',
 'full_address': '   UNIT  4       16     GLADSTONE PARADE GLENROY VIC 3046',
 'LONGITUDE': '144.913823',
 'LATITUDE': '-37.706308',
 'score': 75.0,
 'InputAddress': 'Unit 5 16 Fladstone parade Glenroy 3040'}

In [32]:
import data_loader

In [33]:
a, b, c = data_loader.load_gnaf("GNAF")

Loading GNAF data...
Building BM25 index...
Building address lookup...


In [35]:
c['            9     HAGGAR STREET EAGLEHAWK VIC 3556']

{'ADDRESS_DETAIL_PID': 'GAVIC420571497',
 'LONGITUDE': '144.25677374',
 'LATITUDE': '-36.71892548',
 'full_address_clean': '9 HAGGAR STREET EAGLEHAWK VIC 3556'}